# Loading the CERT data and EDA

# Libraries

In [29]:
import pandas as pd
import os

print(os.getcwd())

/Users/matthewpoole/Documents/Masters/Insider Threats


In [30]:
%ls

answers.tar.bz2                 Questions.md
data download instructions.pdf  r6.2/
Data loading and EDA.ipynb      Updated brief.pdf
Lloyds.pdf


# Loading Data

## Email.csv
First take the date, user, to, activity and size columns To get a feel for the temporal spread of the dataset. This will help with decisions about how to split up the data into train/test/validation sets

In [16]:
# code to extract the data
# mkdir -p extracted

# tar -xjvf r6.2.tar.bz2 \
#   --exclude='http.csv' \
#   --exclude='*/http.csv' \
#   -C extracted

dates = pd.read_csv(
    "r6.2/email_DO_NOT_OPEN.csv",
    usecols=["id","date", "user", "to", "activity","size"]
)

dates.head()

,id,date,user,to,activity,size
0,{I1O2-B4EB49RW-7379WSQW},01/02/2010 06:36:41,HDB1666,Louis.Bernard.Garza@dtaa.com,Send,45659
1,{L7E7-V4UX89RR-3036ZDHU},01/02/2010 06:40:02,HDB1666,Hector.Donovan.Bray@dtaa.com,View,34142
2,{S8C2-Q8YX87DJ-0516SIWZ},01/02/2010 06:42:48,HDB1666,Quintessa.O.Farrell@harris.com,Send,1310925
3,{A1V9-O5BL46SW-1708NAEC},01/02/2010 06:45:42,HDB1666,Hector.Donovan.Bray@dtaa.com,View,23043
4,{N6R0-M2EI82DM-5583LSUM},01/02/2010 06:47:07,HDB1666,Hector.Donovan.Bray@dtaa.com,View,25210


In [18]:
dates["date"] = pd.to_datetime(
    dates["date"],
    errors="coerce"
)

dates["calendar_date"] = dates["date"].dt.date
dates["time"] = dates["date"].dt.time

In [19]:
dates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10994957 entries, 0 to 10994956
Data columns (total 8 columns):
 #   Column         Dtype         
---  ------         -----         
 0   id             object        
 1   date           datetime64[ns]
 2   user           object        
 3   to             object        
 4   activity       object        
 5   size           int64         
 6   calendar_date  object        
 7   time           object        
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 671.1+ MB


In [25]:
dates.describe(include='all')

,id,date,user,to,activity,size,calendar_date,time
count,10994957,10994957,10994957,10994957,10994957,1.099496e+07,10994957,10994957
unique,10994957,NaN,4000,1100362,2,NaN,515,51815
top,{I1O2-B4EB49RW-7379WSQW},NaN,DMH0011,Deborah.Madaline.Horton@dtaa.com,View,NaN,2010-02-03,08:51:02
freq,1,NaN,12570,2514,7162491,NaN,31655,392
mean,NaN,2010-09-11 05:54:02.248917504,NaN,NaN,NaN,4.316567e+05,NaN,NaN
min,NaN,2010-01-02 06:36:41,NaN,NaN,NaN,5.222000e+03,NaN,NaN
25%,NaN,2010-05-04 17:32:55,NaN,NaN,NaN,2.460000e+04,NaN,NaN
50%,NaN,2010-09-08 14:41:03,NaN,NaN,NaN,3.236000e+04,NaN,NaN
75%,NaN,2011-01-19 13:24:08,NaN,NaN,NaN,5.502300e+04,NaN,NaN
max,NaN,2011-05-31 20:10:19,NaN,NaN,NaN,1.474347e+07,NaN,NaN


## Loading malicious emails
Need a list of malicious emails, their IDs and the user identity so that we can make sure these emails are included in training and validation sets. THis is important as they are such rare events.

In [28]:
answers_csv = []

for scenario_num in range(1,6):
    directory = f"r6.2/answers/r6.2-{scenario_num}.csv"

    scenarios = pd.read_csv(directory)
    scenarios["scenario"] = scenario_num
    scenarios["filepath"] = filepath

    answers_csv.append(scenarios)

ParserError: Error tokenizing data. C error: Expected 7 fields in line 3, saw 11


In [35]:
import csv
from pathlib import Path

email_rows = []

file_path = Path("r6.2/answers/r6.2-5.csv")

with file_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.reader(file)

    for record_number, row in enumerate(reader, start=1):
        if not row:
            continue

        row_type = row[0].strip().lower()

        if row_type in {"email", "email.csv"}:
            email_rows.append({
                "record_number": record_number,
                "values": row[1:],
            })


print(f"Found {len(email_rows)} email records")

Found 0 email records


In [48]:
emails = pd.read_csv("r6.2/email_DO_NOT_OPEN.csv")

In [38]:
emails.describe(include='all')

,id,date,user,pc,to,cc,bcc,from,activity,size,attachments,content
count,10994957,10994957,10994957,10994957,10994957,5376676,551672,10994957,10994957,1.099496e+07,2591714,10994957
unique,10994957,7727401,4000,4000,1100362,237562,2444,12186,2,NaN,605217,3760026
top,{I1O2-B4EB49RW-7379WSQW},01/21/2010 09:48:54,DMH0011,PC-3875,Deborah.Madaline.Horton@dtaa.com,Galena.Kay.Leonard@dtaa.com,Ursa.Melinda.Bentley@dtaa.com,Galena.Kay.Leonard@dtaa.com,View,NaN,C:\39NjsN0\HVGT7JG9.doc(1739740),"Since it opened, Brockway Mountain Drive has b..."
freq,1,22,12570,12570,2514,4180,1644,8836,7162491,NaN,441,5359
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.316567e+05,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.059639e+06,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.222000e+03,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.460000e+04,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.236000e+04,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.502300e+04,NaN,NaN


In [41]:
emails['content'].loc[emails['activity'] == "Send"].value_counts().head()

content
Since it opened, Brockway Mountain Drive has been recognized nationally and locally in several media outlets for its picturesque qualities, usually in profiles of Keweenaw County, the Upper Peninsula or other scenic drives. The road was constructed by the county road commission, the Works Project Administration (WPA) and the Civilian Conservation Corps (CCC) in 1933. Brockway Mountain was named for David D. Brockway, one of the pioneer residents of the area. It was briefly used as a connection for the parallel state highway after it opened.    1884
The road was constructed by the county road commission, the Works Project Administration (WPA) and the Civilian Conservation Corps (CCC) in 1933. Brockway Mountain was named for David D. Brockway, one of the pioneer residents of the area.                                                                                                                                                                                                       

In [43]:
emails['activity'].loc[emails['activity'] == "Send"].value_counts() + 
emails['activity'].loc[emails['activity'] == "View"].value_counts()

activity
Send   NaN
View   NaN
Name: count, dtype: float64

activity
View    7162491
Send    3832466
Name: count, dtype: int64

In [ ]:
emails = emails.loc[emails['activity'] == 'Send']

# Exploring embeddings
First only look at the sent emails (all emails are sent or viewed)
Then just look at the first month of activity

In [49]:
# Show that all emails are either send or view, nothing else
emails['activity'].value_counts(dropna=False)

# Filter the emails for only sent ones so that can attribute topics/style to 
emails = emails.loc[emails['activity'] == 'Send']

In [51]:
# All users send emails
emails['user'].value_counts()

user
DMH0011    4426
LSL0001    4355
GKL0006    4137
BVC0009    3577
ACH0008    3354
           ... 
MWH0347      13
NDM2251      12
WMM0873      10
GCM1580       8
LKF0335       8
Name: count, Length: 4000, dtype: int64